# WEEK 9 — LangGraph Foundations

Welcome to Week 9! This notebook is your hands-on companion to the LangGraph Foundations session.

By the end of this notebook you will be able to:
- Explain why LangGraph exists and what problem it solves
- Describe the core building blocks: **Nodes**, **Edges**, and **Shared State**
- Design a simple state schema
- Build and run your first LangGraph workflow (`Query → Retrieval → LLM → Answer`)
- Debug a graph by tracing state changes step by step
- Compare LangGraph with `AgentExecutor` and know when to use which

> 💡 This is a **simple, first-pass version**. We will improvise and add more advanced patterns (conditional edges, loops, multi-agent graphs) in later iterations.

## 🛠️ 0. Setup

We only need three packages for this notebook:
- `langgraph` – the orchestration library itself
- `langchain-openai` – to call an LLM inside our nodes
- `python-dotenv` – to load our API key safely from a `.env` file

## 1️⃣ Why LangGraph Exists

Until now you've built things with **chains** (LCEL) and **AgentExecutor**. Both work, but they start to creak once a workflow gets non-trivial:

| Limitation | Plain Chains | AgentExecutor |
|---|---|---|
| Flow shape | Strictly linear (A → B → C) | The LLM decides the next step at every turn |
| Visibility into intermediate steps | Hard — you mostly see the final output | Better, but still a black-box loop |
| Branching / looping | Not supported natively | Possible, but implicit and hard to control |
| Debugging a wrong step | You re-run the whole chain and guess | You read verbose logs and guess |
| Determinism | High (but inflexible) | Low (LLM decides control flow) |

**The core idea behind LangGraph:**
> Separate *"what the LLM reasons about"* from *"how the application is structured."*

LangGraph lets **you** design the deterministic skeleton of the workflow (the graph), while the **LLM** only handles the parts that genuinely need reasoning (inside individual nodes). This gives you the control of a chain with the flexibility of an agent.

## 2️⃣ Core LangGraph Concepts

A LangGraph workflow has 4 building blocks:

| Concept | What it is |
|---|---|
| **Graph** | The overall workflow — a set of steps and the order they can run in |
| **Node** | A Python function that does *one* job: reads the current state, does some work, returns updates |
| **Edge** | A connection that says "after this node, go to that node" |
| **State** | A shared object (usually a `dict` / `TypedDict`) that every node can read from and write to |

Visually, the simplest possible graph looks like this:

```
User Input
   ↓
Node A
   ↓
Node B
   ↓
Final Response
```

Every arrow above is an **edge**. Every box is a **node**. The thing flowing down the arrows is the **state**.

## 3️⃣ Understanding Graph State

State is just data that's passed from node to node. Each node:
1. Receives the **current state**
2. Does some work
3. Returns a **dictionary with only the keys it wants to update**

LangGraph merges that returned dictionary back into the overall state.

For our first workflow, we'll use this schema (same one from the session slides):

```
{
 query,
 retrieved_docs,
 reasoning_steps,
 final_answer
}
```

Let's define it as a `TypedDict`:

## 4️⃣ Building Your First LangGraph Workflow

We'll build the exact flow from the agenda:

```
User Query → Retrieval → LLM → Answer
```

To keep things simple, our "retriever" is just a small in-memory list of facts about LangGraph — no vector DB needed yet. We'll plug in a real retriever in a later iteration.

### Define the Nodes

A node is just a function that takes the `GraphState` and returns a **partial update**.

### Define the Edges and Compile the Graph

### Run the Graph

## 5️⃣ Debugging Graph Execution

`graph.invoke()` only shows you the *final* state. To debug, use `graph.stream()` — it shows you the state **after every node runs**, which is exactly the "execution trace" we discussed in the session.

### 🐛 A Common Bug: Losing State Instead of Updating It

Here's a mistake almost everyone makes when they start with LangGraph: **overwriting a list instead of appending to it.**

Below, `broken_retriever_node` *replaces* `reasoning_steps` with a brand-new single-item list every time, instead of appending to what's already there. Watch what happens to our debug log.

**The fix:** always build the new list from the *current* state, like we did in the original `retriever_node` / `generator_node`:

```python
"reasoning_steps": state["reasoning_steps"] + ["retriever_node: fetched relevant docs"]
```

This is the most common "incorrect state passing" bug you'll hit — keep it in mind during the live debugging exercise.

## 6️⃣ LangGraph vs AgentExecutor

| | **AgentExecutor** | **LangGraph** |
|---|---|---|
| Control flow | Decided by the LLM at runtime | Decided by you, the developer (graph structure) |
| Determinism | Low — same input can take different paths | High — the path is explicit, only node *content* varies |
| Debuggability | Verbose logs, hard to pinpoint failures | Inspect state after every single node |
| Branching / loops | Implicit, via repeated LLM tool calls | Explicit edges (including conditional edges, covered next week) |
| Best for | Quick prototypes, simple tool-calling agents | Production workflows, multi-step pipelines, multi-agent systems |

**Rule of thumb:**
- If your workflow is a fixed sequence of steps → a **simple chain** is enough.
- If the LLM needs to freely decide which tool to call next, with low stakes → **AgentExecutor** is fine.
- If you need **reliability, visibility, and control** over a multi-step process (RAG pipelines, multi-agent systems, anything going to production) → **LangGraph**.

## 🎯 Try It Yourself

Extend the working graph (not the broken one!) with a **third node** called `critique_node` that:
1. Runs after `generator_node`
2. Asks the LLM: *"Does this answer fully address the question? Reply YES or NO."*
3. Appends its verdict to `reasoning_steps`

```python
def critique_node(state: GraphState) -> dict:
    # TODO: implement me
    ...
```

Wire it in with:
```python
builder.add_edge("generator_node", "critique_node")
builder.add_edge("critique_node", END)
```

(Hint: you'll need to remove the old `generator_node → END` edge first.)

## ✅ Key Takeaways

- **LangGraph** exists to give you deterministic, debuggable control over multi-step LLM workflows.
- A graph is just **Nodes** (functions) + **Edges** (connections) + **State** (shared data).
- Nodes return **partial updates** — always build new lists/dicts from the *current* state to avoid silently losing data.
- `graph.stream()` is your best friend for debugging — it shows you state after every node.
- Use LangGraph when you need control and visibility; use simple chains or AgentExecutor when the workflow is simple enough that you don't.

> Next iteration: conditional edges, loops, and multi-agent graphs. 🚀